In [1]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm
from torch_geometric import utils

from PyTorch_SciPy_interface import PyTorch_SciPy_interface
from FEniCSx_PyTorch_interface import batched_loss_fn

loss_fn = batched_loss_fn()


In [25]:
tset_dir = "training_set_v2"
tset = graph_dataset(f"data/{tset_dir}/input_values")


loader = train_loader(batch_size=1, set=tset)


In [31]:
class mha(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gatv2 = model = tg.nn.models.GAT(
            in_channels=7,
            hidden_channels=5,
            num_layers=10,
            out_channels=1, 
            v2 = True,
            edge_dim=2,
            add_self_loops=False
        )
    def forward(self, data) -> torch.Tensor:
        x, edge_index, edge_attr  = data.x, data.edge_index, data.edge_attr
        h = self.gatv2(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr
        )
        return torch.abs(h)



model = mha()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer)

In [33]:

for i in range(5000):
    print(i)
    loss = self_supervised_train(model=model, loader=loader, loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    print(loss)
    scheduler.step(loss)
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/mha_self_supervised_2.pth")

0
0.10911931845021688
1
0.10911886849592892
2
0.10911656328784147
3
0.10911635201008804
4
0.10911187420390434
5
0.10911178072469015
6
0.10910635259064995
7
0.10910924102698387
8
0.10910466018371133
9
0.1091009768969839
10
0.10910097040346561
11
0.10909953820224287
12
0.10909459035635584
13
0.1090909847647481
14
0.10909219723750786
15
0.10908643011992002
16
0.1090873891576042
17
0.10908360269878482
18
0.10908457808024898
19
0.10907808554136719
20
0.10907522925532476
21
0.10907572628468643
22
0.10907498361117715
23
0.10907163819572153
24
0.10906734997491246
25
0.10906806050093626
26
0.1090640472107845
27
0.10906449239917038
28
0.10906106501724624
29
0.10905728516384401
30
0.10905469721603822
31
0.1090587034302592
32
0.10905434686548109
33
0.10904866764575032
34
0.10904931991369309
35
0.10904675560659495
36
0.10904527736619746
37
0.109040351026248
38
0.10904179338619067
39
0.109037581531652
40
0.10903714690400855
41
0.1090334861972079
42
0.10903390769679659
43
0.10902920821472022
44
0.109

KeyboardInterrupt: 